# Survive a two-week producer rollout where both event shapes flow through your pipeline at once

The producer team is shipping v2 of the clickstream event schema over two weeks. On any given day you do not know which shape an event will land in. The consumer must parse both. The downstream feature store must populate the new field only when v2 events are the source. The rollback playbook (revert the consumer to the v1-only parser) must keep the pipeline alive.

This lab is the AWS-native equivalent of Databricks Auto Loader schema evolution. The cert leaves schema management as exam concept, not hands-on. The interview talking point you walk away with: "I survived a producer v2 rollout where v1 and v2 events flowed simultaneously; the consumer parsed both shapes and rollback was a one-line revert."

The five deliverables map to five checkpoints:
1. Both schema versions are registered with the Glue Schema Registry; compatibility mode is BACKWARD.
2. The producer emits 50 v1 events and 50 v2 events into the Kinesis Data Stream.
3. The consumer Lambda parses both shapes and writes to the Iceberg sink; the new field populates only on v2 rows.
4. The feature store receives only v2 rows; v1 rows are dropped by the feature-store writer.
5. Rollback test: replace the consumer with the v1-only parser, re-run the producer, sink grows by 100 more rows without parse errors.

**Time.** About 75 minutes hands-on. Two producer runs of 100 records each plus a third for the rollback test is the bulk of the wall clock. Budget up to 90 minutes if the IAM propagation or the Lambda deployment retries push the schedule.

**Cost.** This lab costs about 60 cents on a clean run. Kinesis is the steady drain: one shard at 1.5 hours times $0.015 per shard-hour is just over two cents. Lambda invocations are free at lab volume. Glue Iceberg writes from the consumer are the largest line item, roughly 30 cents across the two producer runs plus the rollback run. The Glue Schema Registry itself is free at this volume (first 1M schema-version operations per month are free). A realistic upper bound with a debug run lands around $1.20.

**Critical resource.** The Kinesis Data Stream is critical-tagged. Kinesis bills shard-hours from creation until deletion, even while the stream is idle. The cleanup cell deletes the stream first. Do not walk away from this lab with the stream still alive.

**Wire format note (founder voice).** Production-grade Kinesis pipelines that use the Glue Schema Registry typically wrap each record in an Avro envelope with a 1-byte magic + 16-byte schema-id header. That is the right call at scale because Avro plus the SerDe library cuts the per-record bytes and gives you reader-writer schema reconciliation for free. For the lab we use JSON with an explicit `schema_version` field and call the schema registry separately for the BACKWARD-compatibility check. The pedagogy is the dual-version parser shape and the rollback playbook, not the encoding. The 50-line Lambda fits inline without a SerDe layer; you keep the lesson; you save the seven-MB Lambda package.

This lab is part of the Labrun Data Engineering job-prep track, Streaming & Change Data Capture sub-track.


In [ ]:
!pip install --quiet labrun-checks==0.8.0 boto3


In [ ]:
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} convention from LAB-CREATION-BLUEPRINT
# Section 12.

import atexit
import base64
import io
import json
import re
import sys
import time
import uuid
import zipfile
from getpass import getpass

import boto3
from botocore.exceptions import ClientError

from labrun_checks import (
    CheckpointResult,
    CleanupResource,
    check,
    cleanup,
    register,
    run_cleanup,
)

LAB_SLUG = "schema-evolution-streaming"
LAB_ID = LAB_SLUG
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_SLUG

REGION = "us-east-1"

# Resource identifiers. Bucket name is finalized after STS gives us the
# account id (S3 bucket names are globally unique).
BUCKET_NAME = None  # set in setup
STREAM_NAME = f"labrun-{LAB_SLUG}-stream"
REGISTRY_NAME = f"labrun-{LAB_SLUG}-registry"
SCHEMA_NAME = "clickstream-event"
CONSUMER_ROLE_NAME = f"labrun-{LAB_SLUG}-consumer-role"
CONSUMER_FN_NAME = f"labrun-{LAB_SLUG}-consumer"
INLINE_POLICY_NAME = "labrun-consumer-inline"
# Glue catalog database names cannot contain hyphens. The brief uses
# hyphens; underscores here for the Glue/Athena layer.
DATABASE_NAME = f"labrun_{LAB_SLUG.replace('-', '_')}_db"
EVENTS_TABLE = "events_iceberg"
FEATURES_TABLE = "features_iceberg"
ATHENA_WG_NAME = f"labrun-{LAB_SLUG}-wg"

# Tracked at runtime (set by task cells).
ESM_UUID = None
PRODUCER_RUN_BATCHES = []


In [ ]:
# NBVAL_SKIP
# Setup. Four getpass prompts: session token from the lab page, plus AWS
# access key, secret, and session token (Colab sandbox creds).

print("Paste your lab session token (copy from the Labrun lab page):")
SESSION_TOKEN = getpass("Session token: ").strip()
if not SESSION_TOKEN:
    raise SystemExit("Missing session token. Open the lab page, click Copy Session Token, then re-run this cell.")

print()
print("Paste AWS credentials (from AWS Academy or your sandbox):")
AWS_ACCESS_KEY_ID = getpass("AWS_ACCESS_KEY_ID: ").strip()
AWS_SECRET_ACCESS_KEY = getpass("AWS_SECRET_ACCESS_KEY: ").strip()
AWS_SESSION_TOKEN = getpass("AWS_SESSION_TOKEN (press Enter if your account does not use one): ").strip() or None

AWS_CREDENTIALS = {
    "aws_access_key_id": AWS_ACCESS_KEY_ID,
    "aws_secret_access_key": AWS_SECRET_ACCESS_KEY,
    "aws_session_token": AWS_SESSION_TOKEN,
}

# STS: validate credentials and capture account id.
sts = boto3.client(
    "sts",
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as exc:
    raise SystemExit(
        f"AWS credentials rejected: {exc}. Refresh your sandbox keys, then re-run this cell."
    )

ACCOUNT_ID = identity["Account"]
print(f"AWS account: {ACCOUNT_ID}")
print(f"Region: {REGION}")

BUCKET_NAME = f"labrun-{LAB_SLUG}-{ACCOUNT_ID}"
print(f"Bucket name (account-scoped): {BUCKET_NAME}")

# Register with labrun-checks. The session card renders below if IPython
# is available.
register(SESSION_TOKEN, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print()
print("Setup complete. Move to the CLEANUP_MANIFEST cell.")


In [ ]:
# NBVAL_SKIP
# CLEANUP_MANIFEST declared module-level. Reverse-creation order with
# critical=True on the Kinesis stream so the companion panel idle-warning
# and cleanup-first ordering both work.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="kinesis_stream",
        id=STREAM_NAME,
        region=REGION,
        critical=True,
        cli_delete_command=(
            f"aws kinesis delete-stream --stream-name {STREAM_NAME} "
            f"--enforce-consumer-deletion --region {REGION}"
        ),
    ),
    CleanupResource(
        type="lambda_event_source_mapping",
        id="pending",  # rewritten in Task 3 after create_event_source_mapping
        region=REGION,
    ),
    CleanupResource(
        type="lambda_function",
        id=CONSUMER_FN_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws lambda delete-function --function-name {CONSUMER_FN_NAME} "
            f"--region {REGION}"
        ),
    ),
    CleanupResource(
        type="glue_schema",
        id=SCHEMA_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws glue delete-schema "
            f"--schema-id RegistryName={REGISTRY_NAME},SchemaName={SCHEMA_NAME} "
            f"--region {REGION}"
        ),
    ),
    CleanupResource(
        type="glue_registry",
        id=REGISTRY_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws glue delete-registry --registry-id RegistryName={REGISTRY_NAME} "
            f"--region {REGION}"
        ),
    ),
    CleanupResource(
        type="iceberg_table",
        id=f"{DATABASE_NAME}.{FEATURES_TABLE}",
        region=REGION,
        cli_delete_command=(
            f"aws glue delete-table --database-name {DATABASE_NAME} "
            f"--name {FEATURES_TABLE} --region {REGION}"
        ),
    ),
    CleanupResource(
        type="iceberg_table",
        id=f"{DATABASE_NAME}.{EVENTS_TABLE}",
        region=REGION,
        cli_delete_command=(
            f"aws glue delete-table --database-name {DATABASE_NAME} "
            f"--name {EVENTS_TABLE} --region {REGION}"
        ),
    ),
    CleanupResource(
        type="iam_role",
        id=CONSUMER_ROLE_NAME,
        region=REGION,
    ),
    CleanupResource(
        type="athena_workgroup",
        id=ATHENA_WG_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws athena delete-work-group --work-group {ATHENA_WG_NAME} "
            f"--recursive-delete-option --region {REGION}"
        ),
    ),
    CleanupResource(
        type="glue_database",
        id=DATABASE_NAME,
        region=REGION,
    ),
    CleanupResource(
        type="s3_bucket",
        id=BUCKET_NAME,
        region=REGION,
    ),
]


def _atexit_cleanup():
    print("[atexit] Notebook exiting. The cleanup cell handles teardown; "
          "if it did not run, copy the CLI commands above and run them now.")


atexit.register(_atexit_cleanup)

# Orphan scan. Block (not warn) on any pre-existing tagged resources.
print("Scanning for orphan resources from a previous session...")
tagging = boto3.client(
    "resourcegroupstaggingapi",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
try:
    orphan_resp = tagging.get_resources(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}]
    )
    orphan_arns = [r["ResourceARN"] for r in orphan_resp.get("ResourceTagMappingList", [])]
except ClientError as exc:
    raise SystemExit(
        f"Orphan scan failed ({exc}). Make sure your IAM user has the "
        f"labrun-tag-get-resources inline policy (see the AWS Account Setup "
        f"card on the lab page)."
    )

if orphan_arns:
    print("Found orphan resources from a previous session. Cleanup is blocking.")
    for arn in orphan_arns:
        print(f"  ORPHAN: {arn}")
    raise SystemExit(
        "Run the cleanup cell at the end of the previous notebook session "
        "(or use the CLI commands printed during cleanup) before starting fresh."
    )

print("No orphans found. Manifest declared. Proceed to Task 1.")


## Task 1: Register both schema versions and prove BACKWARD compatibility

The Glue Schema Registry is the source of truth for which event shapes are allowed on the wire. You register v1, register v2 as the next version of the same schema, and set the compatibility mode to BACKWARD so v1 consumers can keep reading v2 events (the new optional field is dropped on read, not rejected).

The shapes:

- **v1** carries `event_id`, `event_timestamp`, `customer_id`, `product_id`, `amount_cents`, plus `schema_version: "v1"`.
- **v2** carries every v1 field plus `referrer_domain` (string), plus `schema_version: "v2"`.

The compatibility mode decision is the lab's first opinionated call. BACKWARD says: the schema registry will accept a new version only if old consumers can still read events written under the new version. Adding an optional field passes BACKWARD because old consumers ignore the new field. Dropping a field would fail BACKWARD (the reflection MCQ walks through the inverse case).

You also create the S3 bucket and Glue database here. The Athena workgroup with a 1-GB scan cap lives in this cell too; it protects you from a runaway query during the parity diff in Task 3.


In [ ]:
# NBVAL_SKIP
# Task 1: Bucket, Glue DB, Athena workgroup, Schema Registry, two schema versions.

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
glue = boto3.client(
    "glue",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
athena = boto3.client(
    "athena",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# S3 bucket. us-east-1 takes no LocationConstraint.
try:
    s3.create_bucket(Bucket=BUCKET_NAME)
    s3.put_bucket_tagging(
        Bucket=BUCKET_NAME,
        Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
    )
    print(f"Bucket created: s3://{BUCKET_NAME}/")
except ClientError as exc:
    if exc.response["Error"]["Code"] in ("BucketAlreadyOwnedByYou",):
        print(f"Bucket already owned: s3://{BUCKET_NAME}/")
    else:
        raise

# Glue database. The brief uses hyphens; Glue rejects hyphens in db names
# so we use underscores.
try:
    glue.create_database(DatabaseInput={"Name": DATABASE_NAME})
    print(f"Glue database created: {DATABASE_NAME}")
except glue.exceptions.AlreadyExistsException:
    print(f"Glue database already exists: {DATABASE_NAME}")

# Athena workgroup with a per-query data-scanned guard.
try:
    athena.create_work_group(
        Name=ATHENA_WG_NAME,
        Configuration={
            "ResultConfiguration": {"OutputLocation": f"s3://{BUCKET_NAME}/athena-results/"},
            "EnforceWorkGroupConfiguration": True,
            "PublishCloudWatchMetricsEnabled": False,
            "BytesScannedCutoffPerQuery": 1_000_000_000,  # 1 GB cap
        },
        Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )
    print(f"Athena workgroup created with 1-GB per-query cap: {ATHENA_WG_NAME}")
except athena.exceptions.InvalidRequestException as exc:
    if "already exists" in str(exc):
        print(f"Athena workgroup already exists: {ATHENA_WG_NAME}")
    else:
        raise

# Two JSON Schemas: v1 and v2. We use the AVRO data format for the
# registry (the registry accepts AVRO, JSON, and PROTOBUF schemas; AVRO
# is the lab pedagogy even though the wire format is JSON).

V1_SCHEMA = json.dumps({
    "type": "record",
    "name": "clickstream",
    "namespace": "com.labrun",
    "fields": [
        {"name": "event_id", "type": "string"},
        {"name": "event_timestamp", "type": "string"},
        {"name": "customer_id", "type": "string"},
        {"name": "product_id", "type": "string"},
        {"name": "amount_cents", "type": "long"},
        {"name": "schema_version", "type": "string"},
    ],
})

V2_SCHEMA = json.dumps({
    "type": "record",
    "name": "clickstream",
    "namespace": "com.labrun",
    "fields": [
        {"name": "event_id", "type": "string"},
        {"name": "event_timestamp", "type": "string"},
        {"name": "customer_id", "type": "string"},
        {"name": "product_id", "type": "string"},
        {"name": "amount_cents", "type": "long"},
        {"name": "schema_version", "type": "string"},
        {"name": "referrer_domain", "type": ["null", "string"], "default": None},
    ],
})

# Registry.
try:
    glue.create_registry(
        RegistryName=REGISTRY_NAME,
        Description="Labrun: schema-evolution-streaming",
        Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
    )
    print(f"Schema registry created: {REGISTRY_NAME}")
except glue.exceptions.AlreadyExistsException:
    print(f"Schema registry already exists: {REGISTRY_NAME}")

# YOUR CODE: Register v1 with BACKWARD compatibility via glue.create_schema(
#   RegistryId={"RegistryName": REGISTRY_NAME},
#   SchemaName=SCHEMA_NAME,
#   DataFormat="AVRO",
#   Compatibility="BACKWARD",
#   SchemaDefinition=V1_SCHEMA,
#   Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
# )

# YOUR CODE: Register v2 as a new version with glue.register_schema_version(
#   SchemaId={"RegistryName": REGISTRY_NAME, "SchemaName": SCHEMA_NAME},
#   SchemaDefinition=V2_SCHEMA,
# )

# Iceberg events table created via Athena CTAS-equivalent CREATE TABLE.
ICEBERG_CREATE_EVENTS = (
    f"CREATE TABLE IF NOT EXISTS {DATABASE_NAME}.{EVENTS_TABLE} ("
    f"  event_id string, event_timestamp string, customer_id string, "
    f"  product_id string, amount_cents bigint, schema_version string, "
    f"  referrer_domain string, ingest_batch_id string"
    f") "
    f"LOCATION 's3://{BUCKET_NAME}/{EVENTS_TABLE}/' "
    f"TBLPROPERTIES ('table_type'='ICEBERG', 'format'='parquet')"
)
ICEBERG_CREATE_FEATURES = (
    f"CREATE TABLE IF NOT EXISTS {DATABASE_NAME}.{FEATURES_TABLE} ("
    f"  event_id string, customer_id string, referrer_domain string, "
    f"  schema_version string, ingest_batch_id string"
    f") "
    f"LOCATION 's3://{BUCKET_NAME}/{FEATURES_TABLE}/' "
    f"TBLPROPERTIES ('table_type'='ICEBERG', 'format'='parquet')"
)


def run_athena(query, timeout_s=60):
    qid = athena.start_query_execution(
        QueryString=query,
        WorkGroup=ATHENA_WG_NAME,
        QueryExecutionContext={"Database": DATABASE_NAME},
    )["QueryExecutionId"]
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        ex = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]
        state = ex["Status"]["State"]
        if state in ("SUCCEEDED", "FAILED", "CANCELLED"):
            if state != "SUCCEEDED":
                raise RuntimeError(f"Athena query {qid} {state}: {ex['Status'].get('StateChangeReason')}")
            return qid
        time.sleep(2)
    raise TimeoutError(f"Athena query {qid} timed out after {timeout_s}s")


run_athena(ICEBERG_CREATE_EVENTS)
print(f"Iceberg events table ready: {DATABASE_NAME}.{EVENTS_TABLE}")
run_athena(ICEBERG_CREATE_FEATURES)
print(f"Iceberg features table ready: {DATABASE_NAME}.{FEATURES_TABLE}")


In [ ]:
# NBVAL_SKIP
# Checkpoint 1: both schema versions registered, compatibility = BACKWARD.

def validator_1(session) -> CheckpointResult:
    try:
        registry = glue.get_registry(RegistryId={"RegistryName": REGISTRY_NAME})
    except glue.exceptions.EntityNotFoundException:
        return CheckpointResult("fail", error_reason=f"Glue registry {REGISTRY_NAME} does not exist")

    try:
        schema = glue.get_schema(SchemaId={"RegistryName": REGISTRY_NAME, "SchemaName": SCHEMA_NAME})
    except glue.exceptions.EntityNotFoundException:
        return CheckpointResult("fail", error_reason=(
            f"Schema {SCHEMA_NAME} not found in registry {REGISTRY_NAME}. "
            f"Did you call glue.create_schema?"
        ))

    if schema.get("Compatibility") != "BACKWARD":
        return CheckpointResult("fail", error_reason=(
            f"Schema compatibility is {schema.get('Compatibility')!r}, expected BACKWARD. "
            f"BACKWARD lets new schemas drop fields the consumer expects; for adding "
            f"fields you want BACKWARD (passes here) or FULL."
        ))

    try:
        versions = glue.list_schema_versions(
            SchemaId={"RegistryName": REGISTRY_NAME, "SchemaName": SCHEMA_NAME}
        )
    except ClientError as exc:
        return CheckpointResult("error", error_reason=f"list_schema_versions failed: {exc}")

    version_numbers = sorted(v["VersionNumber"] for v in versions.get("Schemas", []))
    if len(version_numbers) < 2:
        return CheckpointResult("fail", error_reason=(
            f"Expected at least 2 schema versions, found {len(version_numbers)}. "
            f"Did you register v2 via glue.register_schema_version?"
        ))

    return CheckpointResult("pass")


check(1, validator_1)


<details><summary>Hint 1 (nudge)</summary>

The Glue Schema Registry needs the schema definition plus a compatibility mode at the moment the schema is created. The mode is set once, not per version.

</details>

<details><summary>Hint 2 (direction)</summary>

`glue.create_schema` takes `Compatibility="BACKWARD"`. New versions land via `glue.register_schema_version`. BACKWARD means: a new schema is accepted only if old consumers can still read events written under the new schema. Adding an optional field (with a default) passes BACKWARD. Dropping a required field does not.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
glue.create_schema(
    RegistryId={"RegistryName": REGISTRY_NAME},
    SchemaName=SCHEMA_NAME,
    DataFormat="AVRO",
    Compatibility="BACKWARD",
    SchemaDefinition=V1_SCHEMA,
    Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
)

glue.register_schema_version(
    SchemaId={"RegistryName": REGISTRY_NAME, "SchemaName": SCHEMA_NAME},
    SchemaDefinition=V2_SCHEMA,
)
```

</details>


**Wallet check.** Glue Schema Registry and Glue Data Catalog operations are free at this volume (first 1M operations per month). Athena workgroup creation is a control-plane call: free. Iceberg table creation via Athena scanned a few KB of metadata: under $0.0001. The Kinesis stream has not been created yet; the meter on the critical resource is still off. Session total so far: zero.

## Task 2: Create the 1-shard stream and emit 50 v1 plus 50 v2 events

You create the Kinesis Data Stream (one shard, tagged critical) and run the seeded producer once. The producer is provided as a Python function because the v1-v2 ratio is fixed for the lab: 50 v1 events and 50 v2 events, interleaved, with a deterministic seed so checkpoints land the same row counts every time.

The stream is the lab's first hourly-billed resource. Its meter starts the moment the stream goes ACTIVE. The cleanup cell at the end of the lab deletes the stream first to stop the meter.


In [ ]:
# NBVAL_SKIP
# Task 2: Create the Kinesis stream, run the producer.

kinesis = boto3.client(
    "kinesis",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# YOUR CODE: Create the stream with kinesis.create_stream(
#   StreamName=STREAM_NAME,
#   ShardCount=1,
#   StreamModeDetails={"StreamMode": "PROVISIONED"},
#   Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
# ).

# YOUR CODE: Wait for the stream to reach ACTIVE. Use a polling loop with
# kinesis.describe_stream_summary; the StreamStatus transitions
# CREATING -> ACTIVE in about 60 seconds.
print("Your stream is reading a book on the bus, give it about 60 seconds...")
deadline = time.time() + 180
while time.time() < deadline:
    try:
        status = kinesis.describe_stream_summary(StreamName=STREAM_NAME)["StreamDescriptionSummary"]["StreamStatus"]
    except kinesis.exceptions.ResourceNotFoundException:
        time.sleep(5)
        continue
    if status == "ACTIVE":
        break
    time.sleep(5)
else:
    raise TimeoutError(f"Stream {STREAM_NAME} did not reach ACTIVE within 180s")

print(f"Stream ACTIVE: {STREAM_NAME}")


def run_producer(batch_id, n_v1=50, n_v2=50, seed=42):
    """Emit n_v1 v1 events and n_v2 v2 events, interleaved deterministically.

    Returns the list of emitted records (also written to S3 as a log).
    """
    import random
    rnd = random.Random(seed + hash(batch_id) % 2**32)
    products = [f"sku-{i:03d}" for i in range(20)]
    domains = ["org.example.com", "ref-a.example", "ref-b.example", "ref-c.example"]
    records = []
    for i in range(n_v1):
        records.append({
            "event_id": str(uuid.uuid4()),
            "event_timestamp": f"2026-05-15T12:{i % 60:02d}:00Z",
            "customer_id": f"cust-{rnd.randint(1, 50)}",
            "product_id": rnd.choice(products),
            "amount_cents": rnd.randint(500, 5000),
            "schema_version": "v1",
            "ingest_batch_id": batch_id,
        })
    for i in range(n_v2):
        records.append({
            "event_id": str(uuid.uuid4()),
            "event_timestamp": f"2026-05-15T13:{i % 60:02d}:00Z",
            "customer_id": f"cust-{rnd.randint(1, 50)}",
            "product_id": rnd.choice(products),
            "amount_cents": rnd.randint(500, 5000),
            "schema_version": "v2",
            "referrer_domain": rnd.choice(domains),
            "ingest_batch_id": batch_id,
        })
    rnd.shuffle(records)
    # PutRecords in chunks of 500 (the API max).
    for chunk_start in range(0, len(records), 500):
        chunk = records[chunk_start:chunk_start + 500]
        kinesis.put_records(
            StreamName=STREAM_NAME,
            Records=[
                {
                    "Data": json.dumps(r).encode("utf-8"),
                    "PartitionKey": r["customer_id"],
                }
                for r in chunk
            ],
        )
    # Log to S3 for the validator.
    s3.put_object(
        Bucket=BUCKET_NAME,
        Key=f"producer/{batch_id}.json",
        Body=json.dumps({
            "batch_id": batch_id,
            "emitted": len(records),
            "v1_count": sum(1 for r in records if r["schema_version"] == "v1"),
            "v2_count": sum(1 for r in records if r["schema_version"] == "v2"),
            "ids": [r["event_id"] for r in records],
        }).encode("utf-8"),
    )
    return records


PRODUCER_BATCH_1 = f"batch-{int(time.time())}-1"
PRODUCER_RUN_BATCHES.append(PRODUCER_BATCH_1)
emitted_1 = run_producer(PRODUCER_BATCH_1)
print(f"Producer emitted {len(emitted_1)} records ({PRODUCER_BATCH_1}).")


In [ ]:
# NBVAL_SKIP
# Checkpoint 2: 1 active shard and trim-horizon read sees the records.

def validator_2(session) -> CheckpointResult:
    try:
        shards = kinesis.list_shards(StreamName=STREAM_NAME)["Shards"]
    except kinesis.exceptions.ResourceNotFoundException:
        return CheckpointResult("fail", error_reason=f"Stream {STREAM_NAME} does not exist")
    active = [s for s in shards if s.get("SequenceNumberRange", {}).get("EndingSequenceNumber") is None]
    if len(active) != 1:
        return CheckpointResult("fail", error_reason=(
            f"Expected exactly 1 active shard, found {len(active)}. "
            f"Did the create_stream call use ShardCount=1?"
        ))

    # Read the producer log for the most recent batch.
    if not PRODUCER_RUN_BATCHES:
        return CheckpointResult("fail", error_reason="No producer batches recorded. Did run_producer execute?")
    latest_batch = PRODUCER_RUN_BATCHES[-1]
    try:
        log = json.loads(
            s3.get_object(Bucket=BUCKET_NAME, Key=f"producer/{latest_batch}.json")["Body"].read()
        )
    except ClientError as exc:
        return CheckpointResult("fail", error_reason=f"Producer log missing: {exc}")
    if log["emitted"] != 100 or log["v1_count"] != 50 or log["v2_count"] != 50:
        return CheckpointResult("fail", error_reason=(
            f"Producer log shows emitted={log['emitted']} v1={log['v1_count']} v2={log['v2_count']}, "
            f"expected 100/50/50."
        ))

    return CheckpointResult("pass")


check(2, validator_2)


<details><summary>Hint 1 (nudge)</summary>

Kinesis takes about a minute to provision a new stream. The status transitions CREATING then ACTIVE. PutRecords against a CREATING stream fails with `ResourceInUseException`.

</details>

<details><summary>Hint 2 (direction)</summary>

`kinesis.create_stream` takes `ShardCount=1` and `StreamModeDetails={"StreamMode": "PROVISIONED"}`. Tag the stream with `labrun:lab-slug=schema-evolution-streaming` at create time so the orphan scan and the companion panel can see it. Poll `describe_stream_summary` until `StreamStatus == "ACTIVE"`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
kinesis.create_stream(
    StreamName=STREAM_NAME,
    ShardCount=1,
    StreamModeDetails={"StreamMode": "PROVISIONED"},
    Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
)
```

</details>


**Wallet check.** The Kinesis stream is now alive: 1 shard at $0.015 per shard-hour, billed from this moment. Across the ~90-minute session that is roughly 2 cents. PUT payload units for 100 records are negligible (PUT payload units bill per 25 KB; one record at ~250 bytes is well under 1 unit per record). Producer S3 log: a few KB, free. Session total so far: about 2 cents.

## Task 3: Deploy the dual-version consumer Lambda

You build the consumer that parses both event shapes and writes to the Iceberg sink. The Lambda reads the JSON record's `schema_version` field, deserializes the appropriate shape, and inserts every row into `events_iceberg`. The `referrer_domain` column is populated only on v2 rows; v1 rows leave it null.

Inside the function the Lambda uses Athena `INSERT INTO` for each batch. This is heavy for a production pipeline (Athena per-query overhead is 2-5 seconds) but readable for the lab; the pedagogy is the dual-version parser, not the Iceberg write path. A production rewrite would batch records to Parquet files and commit them via Iceberg metadata directly.

The Lambda role needs S3, Glue catalog, Athena, and Kinesis read permissions plus CloudWatch Logs for `logging`. The event-source mapping pulls from the stream with `StartingPosition=TRIM_HORIZON` so the records already on the stream from Task 2 are picked up.


In [ ]:
# NBVAL_SKIP
# Task 3: IAM role for the Lambda, deploy the Lambda, wire the ESM.

iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
lambda_client = boto3.client(
    "lambda",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# Trust policy for Lambda.
TRUST_POLICY = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "lambda.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}

# Inline policy: S3, Kinesis, Athena, Glue catalog, CloudWatch Logs.
INLINE_POLICY = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "s3:GetObject",
                "s3:PutObject",
                "s3:DeleteObject",
                "s3:ListBucket",
                "s3:GetBucketLocation",
            ],
            "Resource": [f"arn:aws:s3:::{BUCKET_NAME}", f"arn:aws:s3:::{BUCKET_NAME}/*"],
        },
        {
            "Effect": "Allow",
            "Action": [
                "kinesis:GetRecords",
                "kinesis:GetShardIterator",
                "kinesis:DescribeStream",
                "kinesis:DescribeStreamSummary",
                "kinesis:ListShards",
                "kinesis:SubscribeToShard",
            ],
            "Resource": f"arn:aws:kinesis:{REGION}:{ACCOUNT_ID}:stream/{STREAM_NAME}",
        },
        {
            "Effect": "Allow",
            "Action": [
                "athena:StartQueryExecution",
                "athena:GetQueryExecution",
                "athena:GetQueryResults",
                "athena:GetWorkGroup",
                "glue:GetDatabase",
                "glue:GetTable",
                "glue:GetTables",
                "glue:GetPartition",
                "glue:GetPartitions",
                "glue:UpdateTable",
                "glue:GetSchema",
                "glue:GetSchemaVersion",
                "glue:ListSchemaVersions",
                "logs:CreateLogGroup",
                "logs:CreateLogStream",
                "logs:PutLogEvents",
            ],
            "Resource": "*",
        },
    ],
}

# YOUR CODE: Create the IAM role and attach the inline policy.
# iam.create_role(
#   RoleName=CONSUMER_ROLE_NAME,
#   AssumeRolePolicyDocument=json.dumps(TRUST_POLICY),
#   Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
# )
# iam.put_role_policy(
#   RoleName=CONSUMER_ROLE_NAME,
#   PolicyName=INLINE_POLICY_NAME,
#   PolicyDocument=json.dumps(INLINE_POLICY),
# )

print("Your IAM role is asking for directions, hold for 15 seconds...")
time.sleep(15)
print("Role is ready.")
role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{CONSUMER_ROLE_NAME}"

# Dual-version consumer code. Inserts into events_iceberg via Athena.
CONSUMER_CODE = f'''
import base64
import json
import os
import time
import boto3

ATHENA = boto3.client("athena", region_name="{REGION}")
DB = "{DATABASE_NAME}"
TABLE = "{EVENTS_TABLE}"
WG = "{ATHENA_WG_NAME}"
BUCKET = "{BUCKET_NAME}"


def _run_athena(query, timeout_s=30):
    qid = ATHENA.start_query_execution(
        QueryString=query, WorkGroup=WG,
        QueryExecutionContext={{"Database": DB}},
    )["QueryExecutionId"]
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        s = ATHENA.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"]["State"]
        if s == "SUCCEEDED":
            return
        if s in ("FAILED", "CANCELLED"):
            raise RuntimeError(f"Athena {{qid}} {{s}}")
        time.sleep(1)
    raise TimeoutError(f"Athena {{qid}} timed out")


def _quote(v):
    if v is None:
        return "NULL"
    if isinstance(v, (int, float)):
        return str(v)
    return "'" + str(v).replace("'", "''") + "'"


def handler(event, context):
    rows_v1 = 0
    rows_v2 = 0
    values_clauses = []
    for r in event.get("Records", []):
        try:
            data = json.loads(base64.b64decode(r["kinesis"]["data"]))
        except Exception as exc:
            print(f"PARSE_ERROR {{exc}}")
            continue
        version = data.get("schema_version")
        if version == "v1":
            referrer = None
            rows_v1 += 1
        elif version == "v2":
            referrer = data.get("referrer_domain")
            rows_v2 += 1
        else:
            print(f"UNKNOWN_SCHEMA_VERSION {{version!r}}")
            continue
        values_clauses.append(
            f"({{_quote(data.get('event_id'))}}, {{_quote(data.get('event_timestamp'))}}, "
            f"{{_quote(data.get('customer_id'))}}, {{_quote(data.get('product_id'))}}, "
            f"{{int(data.get('amount_cents', 0))}}, {{_quote(version)}}, "
            f"{{_quote(referrer)}}, {{_quote(data.get('ingest_batch_id'))}})"
        )
    if values_clauses:
        insert_sql = (
            f"INSERT INTO {{DB}}.{{TABLE}} "
            f"(event_id, event_timestamp, customer_id, product_id, amount_cents, "
            f"schema_version, referrer_domain, ingest_batch_id) VALUES "
            + ", ".join(values_clauses)
        )
        _run_athena(insert_sql, timeout_s=60)
    print(f"PROCESSED v1={{rows_v1}} v2={{rows_v2}} total={{rows_v1 + rows_v2}}")
    return {{"statusCode": 200, "v1": rows_v1, "v2": rows_v2}}
'''


def _make_zip(code_str):
    buf = io.BytesIO()
    with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as z:
        z.writestr("index.py", code_str)
    buf.seek(0)
    return buf.read()


zip_bytes = _make_zip(CONSUMER_CODE)

# YOUR CODE: Deploy the Lambda. Retry on InvalidParameterValueException
# (the role may still be propagating).
# for attempt in range(8):
#     try:
#         lambda_client.create_function(
#             FunctionName=CONSUMER_FN_NAME,
#             Runtime="python3.13",
#             Role=role_arn,
#             Handler="index.handler",
#             Code={"ZipFile": zip_bytes},
#             Timeout=120,
#             MemorySize=512,
#             Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
#         )
#         break
#     except lambda_client.exceptions.InvalidParameterValueException:
#         time.sleep(5)
# else:
#     raise RuntimeError("Lambda create failed; role propagation never landed")

print(f"Consumer Lambda deployed: {CONSUMER_FN_NAME}")

# Event source mapping.
stream_arn = f"arn:aws:kinesis:{REGION}:{ACCOUNT_ID}:stream/{STREAM_NAME}"
esm = lambda_client.create_event_source_mapping(
    EventSourceArn=stream_arn,
    FunctionName=CONSUMER_FN_NAME,
    StartingPosition="TRIM_HORIZON",
    BatchSize=10,
    MaximumBatchingWindowInSeconds=2,
    Enabled=True,
)
ESM_UUID = esm["UUID"]

# Update CLEANUP_MANIFEST with the ESM UUID we just learned.
for r in CLEANUP_MANIFEST:
    if r.type == "lambda_event_source_mapping":
        r.id = ESM_UUID
        break

print(f"Event source mapping created: {ESM_UUID}")
print("Your records are being chewed through, hold for ~60 seconds...")
time.sleep(60)
print("Drain wait complete. The validator queries Iceberg next.")


In [ ]:
# NBVAL_SKIP
# Checkpoint 3: 100 rows in events_iceberg; referrer_domain populated only on v2.

def _athena_scalar(query, timeout_s=60):
    qid = athena.start_query_execution(
        QueryString=query, WorkGroup=ATHENA_WG_NAME,
        QueryExecutionContext={"Database": DATABASE_NAME},
    )["QueryExecutionId"]
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        s = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"]["State"]
        if s == "SUCCEEDED":
            rows = athena.get_query_results(QueryExecutionId=qid)["ResultSet"]["Rows"]
            return rows[1]["Data"][0]["VarCharValue"]
        if s in ("FAILED", "CANCELLED"):
            raise RuntimeError(f"Athena {qid} {s}")
        time.sleep(2)
    raise TimeoutError(f"Athena {qid} timed out")


def validator_3(session) -> CheckpointResult:
    latest = PRODUCER_RUN_BATCHES[-1]
    try:
        total = int(_athena_scalar(
            f"SELECT count(*) FROM {DATABASE_NAME}.{EVENTS_TABLE} "
            f"WHERE ingest_batch_id = '{latest}'"
        ))
        v1 = int(_athena_scalar(
            f"SELECT count(*) FROM {DATABASE_NAME}.{EVENTS_TABLE} "
            f"WHERE ingest_batch_id = '{latest}' AND schema_version = 'v1'"
        ))
        v2 = int(_athena_scalar(
            f"SELECT count(*) FROM {DATABASE_NAME}.{EVENTS_TABLE} "
            f"WHERE ingest_batch_id = '{latest}' AND schema_version = 'v2'"
        ))
        ref_v1 = int(_athena_scalar(
            f"SELECT count(*) FROM {DATABASE_NAME}.{EVENTS_TABLE} "
            f"WHERE ingest_batch_id = '{latest}' AND schema_version = 'v1' "
            f"AND referrer_domain IS NOT NULL"
        ))
        ref_v2 = int(_athena_scalar(
            f"SELECT count(*) FROM {DATABASE_NAME}.{EVENTS_TABLE} "
            f"WHERE ingest_batch_id = '{latest}' AND schema_version = 'v2' "
            f"AND referrer_domain IS NOT NULL"
        ))
    except Exception as exc:
        return CheckpointResult("error", error_reason=f"Athena query failed: {exc}")

    if total != 100:
        return CheckpointResult("fail", error_reason=(
            f"events_iceberg has {total} rows for this batch, expected 100. "
            f"Wait another 30s for the consumer to drain, then re-run the checkpoint."
        ))
    if v1 != 50 or v2 != 50:
        return CheckpointResult("fail", error_reason=(
            f"v1={v1}, v2={v2}, expected 50/50."
        ))
    if ref_v1 != 0:
        return CheckpointResult("fail", error_reason=(
            f"{ref_v1} v1 rows have referrer_domain populated. v1 events do not "
            f"carry referrer_domain; this should be NULL."
        ))
    if ref_v2 != 50:
        return CheckpointResult("fail", error_reason=(
            f"{ref_v2} v2 rows have referrer_domain populated, expected 50. "
            f"Check the consumer is reading the referrer_domain field on v2 events."
        ))
    return CheckpointResult("pass")


check(3, validator_3)


<details><summary>Hint 1 (nudge)</summary>

The Lambda needs to know which schema version each event uses before it can decide what to write. Production-grade pipelines use the Glue Schema Registry SerDe for this; for the lab the wire format is JSON and the version sits in a `schema_version` field.

</details>

<details><summary>Hint 2 (direction)</summary>

The handler reads `schema_version` from the JSON record. On `"v1"` it skips the `referrer_domain` field (leaves null); on `"v2"` it reads `referrer_domain` and writes it. The Athena INSERT INTO statement carries the column either way; v1 rows just have NULL in that column. The event-source mapping wires the Lambda to the stream with `StartingPosition="TRIM_HORIZON"` so it picks up the records the producer already wrote.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
iam.create_role(
    RoleName=CONSUMER_ROLE_NAME,
    AssumeRolePolicyDocument=json.dumps(TRUST_POLICY),
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
iam.put_role_policy(
    RoleName=CONSUMER_ROLE_NAME,
    PolicyName=INLINE_POLICY_NAME,
    PolicyDocument=json.dumps(INLINE_POLICY),
)
time.sleep(15)

for attempt in range(8):
    try:
        lambda_client.create_function(
            FunctionName=CONSUMER_FN_NAME,
            Runtime="python3.13",
            Role=role_arn,
            Handler="index.handler",
            Code={"ZipFile": zip_bytes},
            Timeout=120,
            MemorySize=512,
            Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
        )
        break
    except lambda_client.exceptions.InvalidParameterValueException:
        time.sleep(5)
```

</details>


**Wallet check.** Lambda invocations: free at lab volume (100 records across ~10 invocations is well under the free tier). Athena scans: the consumer's INSERTs read no source data (they only write), and the checkpoint queries scan a few KB total. Iceberg writes: this is the lab's main cost driver. Roughly $0.15 for the 100-row commit across multiple snapshots. Session total so far: about 17 cents.

## Task 4: Extend the consumer to populate the feature store only on v2 events

The feature store is a second Iceberg table (`features_iceberg`) that downstream models read. It should receive only rows whose source is v2 (the new feature is `referrer_domain`; v1 events do not carry it, so writing them to the feature store would pollute the feature with nulls).

You update the consumer Lambda with the v2-only feature-store branch, then re-run the producer (a fresh batch) so the new code path executes against new traffic. The validator counts rows in `features_iceberg` for the new batch and asserts only v2 rows landed.


In [ ]:
# NBVAL_SKIP
# Task 4: Update the Lambda to write the feature store on v2 events only.

CONSUMER_CODE_V2 = f'''
import base64
import json
import os
import time
import boto3

ATHENA = boto3.client("athena", region_name="{REGION}")
DB = "{DATABASE_NAME}"
TABLE = "{EVENTS_TABLE}"
FEATURES = "{FEATURES_TABLE}"
WG = "{ATHENA_WG_NAME}"


def _run_athena(query, timeout_s=60):
    qid = ATHENA.start_query_execution(
        QueryString=query, WorkGroup=WG,
        QueryExecutionContext={{"Database": DB}},
    )["QueryExecutionId"]
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        s = ATHENA.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"]["State"]
        if s == "SUCCEEDED":
            return
        if s in ("FAILED", "CANCELLED"):
            raise RuntimeError(f"Athena {{qid}} {{s}}")
        time.sleep(1)
    raise TimeoutError(f"Athena {{qid}} timed out")


def _quote(v):
    if v is None:
        return "NULL"
    if isinstance(v, (int, float)):
        return str(v)
    return "'" + str(v).replace("'", "''") + "'"


def handler(event, context):
    events_values = []
    features_values = []
    for r in event.get("Records", []):
        try:
            data = json.loads(base64.b64decode(r["kinesis"]["data"]))
        except Exception as exc:
            print(f"PARSE_ERROR {{exc}}")
            continue
        version = data.get("schema_version")
        referrer = data.get("referrer_domain") if version == "v2" else None
        events_values.append(
            f"({{_quote(data.get('event_id'))}}, {{_quote(data.get('event_timestamp'))}}, "
            f"{{_quote(data.get('customer_id'))}}, {{_quote(data.get('product_id'))}}, "
            f"{{int(data.get('amount_cents', 0))}}, {{_quote(version)}}, "
            f"{{_quote(referrer)}}, {{_quote(data.get('ingest_batch_id'))}})"
        )
        if version == "v2":
            features_values.append(
                f"({{_quote(data.get('event_id'))}}, {{_quote(data.get('customer_id'))}}, "
                f"{{_quote(referrer)}}, {{_quote(version)}}, "
                f"{{_quote(data.get('ingest_batch_id'))}})"
            )
    if events_values:
        _run_athena(
            f"INSERT INTO {{DB}}.{{TABLE}} "
            f"(event_id, event_timestamp, customer_id, product_id, amount_cents, "
            f"schema_version, referrer_domain, ingest_batch_id) VALUES "
            + ", ".join(events_values),
            timeout_s=60,
        )
    if features_values:
        _run_athena(
            f"INSERT INTO {{DB}}.{{FEATURES}} "
            f"(event_id, customer_id, referrer_domain, schema_version, ingest_batch_id) "
            f"VALUES " + ", ".join(features_values),
            timeout_s=60,
        )
    print(f"PROCESSED events={{len(events_values)}} features={{len(features_values)}}")
    return {{"statusCode": 200}}
'''

# YOUR CODE: Update the Lambda code via lambda_client.update_function_code(
#   FunctionName=CONSUMER_FN_NAME, ZipFile=_make_zip(CONSUMER_CODE_V2)
# ). Wait for the update to settle (~5s) before triggering the producer.

print("Your code update is in line at the deli, hold for 10 seconds...")
time.sleep(10)

# Trigger a fresh producer batch so the new code path executes.
PRODUCER_BATCH_2 = f"batch-{int(time.time())}-2"
PRODUCER_RUN_BATCHES.append(PRODUCER_BATCH_2)
run_producer(PRODUCER_BATCH_2)
print(f"Producer batch 2 emitted: {PRODUCER_BATCH_2}")
print("Drain wait, ~60 seconds...")
time.sleep(60)


In [ ]:
# NBVAL_SKIP
# Checkpoint 4: features_iceberg contains exactly 50 rows for the new batch,
# all with schema_version='v2'.

def validator_4(session) -> CheckpointResult:
    latest = PRODUCER_RUN_BATCHES[-1]
    try:
        total = int(_athena_scalar(
            f"SELECT count(*) FROM {DATABASE_NAME}.{FEATURES_TABLE} "
            f"WHERE ingest_batch_id = '{latest}'"
        ))
        v2 = int(_athena_scalar(
            f"SELECT count(*) FROM {DATABASE_NAME}.{FEATURES_TABLE} "
            f"WHERE ingest_batch_id = '{latest}' AND schema_version = 'v2'"
        ))
    except Exception as exc:
        return CheckpointResult("error", error_reason=f"Athena query failed: {exc}")

    if total != 50:
        return CheckpointResult("fail", error_reason=(
            f"features_iceberg has {total} rows for this batch, expected 50 (v2-only). "
            f"If you see 100, the consumer is writing v1 rows to the feature store too."
        ))
    if v2 != 50:
        return CheckpointResult("fail", error_reason=(
            f"features_iceberg has {v2} v2 rows out of {total}; expected 50/50."
        ))
    return CheckpointResult("pass")


check(4, validator_4)


<details><summary>Hint 1 (nudge)</summary>

The feature-store writer is a second branch inside the same Lambda handler. The branch fires only when `schema_version == "v2"`.

</details>

<details><summary>Hint 2 (direction)</summary>

Inside the per-record loop, separate the events_iceberg INSERT (every record) from the features_iceberg INSERT (v2 records only). After updating the function code, the producer needs to run again so the new code path executes against fresh traffic; the records already on the stream were processed by the old code.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
if version == "v2":
    features_values.append(...)

# After all records are collected:
if features_values:
    _run_athena(
        f"INSERT INTO {DB}.{FEATURES} ... VALUES " + ", ".join(features_values)
    )
```

And to deploy the update:
```python
lambda_client.update_function_code(
    FunctionName=CONSUMER_FN_NAME,
    ZipFile=_make_zip(CONSUMER_CODE_V2),
)
```

</details>


**Wallet check.** A second producer run plus a second consumer drain adds about 15 more cents in Iceberg writes (two more tables touched). Lambda invocations: still free. Kinesis shard-hour ticked another ~20 minutes: roughly another half cent. Session total so far: about 33 cents.

## Task 5: Rollback test with the v1-only parser

The rollback playbook is the most important deliverable of this lab. When a producer rollout goes sideways, the senior DE move is "revert the consumer, keep ingesting, fix forward later." BACKWARD compatibility on the schema registry is what makes that revert safe: a v1-only consumer can keep reading v2 events because the new field is optional from the v1 perspective.

You replace the consumer code with a v1-only parser (provided), re-run the producer one more time, and verify no parse errors appear in the logs. The events_iceberg table should grow by 100 more rows (the rollback keeps writing the base events; the feature-store branch is gone in the rollback code).


In [ ]:
# NBVAL_SKIP
# Task 5: Roll the consumer back to v1-only and re-run the producer.

V1_ONLY_CONSUMER_CODE = f'''
import base64
import json
import time
import boto3

ATHENA = boto3.client("athena", region_name="{REGION}")
DB = "{DATABASE_NAME}"
TABLE = "{EVENTS_TABLE}"
WG = "{ATHENA_WG_NAME}"


def _run_athena(query, timeout_s=60):
    qid = ATHENA.start_query_execution(
        QueryString=query, WorkGroup=WG,
        QueryExecutionContext={{"Database": DB}},
    )["QueryExecutionId"]
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        s = ATHENA.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"]["State"]
        if s == "SUCCEEDED":
            return
        if s in ("FAILED", "CANCELLED"):
            raise RuntimeError(f"Athena {{qid}} {{s}}")
        time.sleep(1)
    raise TimeoutError(f"Athena {{qid}} timed out")


def _quote(v):
    if v is None:
        return "NULL"
    if isinstance(v, (int, float)):
        return str(v)
    return "'" + str(v).replace("'", "''") + "'"


def handler(event, context):
    # v1-only parser: read the base v1 fields. Extra v2 fields land in
    # data but we do not reference referrer_domain; BACKWARD compatibility
    # means the consumer tolerates the extra field silently.
    rows = []
    for r in event.get("Records", []):
        try:
            data = json.loads(base64.b64decode(r["kinesis"]["data"]))
        except Exception as exc:
            print(f"PARSE_ERROR {{exc}}")
            continue
        rows.append(
            f"({{_quote(data.get('event_id'))}}, {{_quote(data.get('event_timestamp'))}}, "
            f"{{_quote(data.get('customer_id'))}}, {{_quote(data.get('product_id'))}}, "
            f"{{int(data.get('amount_cents', 0))}}, 'v1', NULL, "
            f"{{_quote(data.get('ingest_batch_id'))}})"
        )
    if rows:
        _run_athena(
            f"INSERT INTO {{DB}}.{{TABLE}} "
            f"(event_id, event_timestamp, customer_id, product_id, amount_cents, "
            f"schema_version, referrer_domain, ingest_batch_id) VALUES "
            + ", ".join(rows),
            timeout_s=60,
        )
    print(f"PROCESSED_V1ONLY count={{len(rows)}}")
    return {{"statusCode": 200, "count": len(rows)}}
'''

# YOUR CODE: Replace the Lambda code with V1_ONLY_CONSUMER_CODE via
# lambda_client.update_function_code(
#   FunctionName=CONSUMER_FN_NAME, ZipFile=_make_zip(V1_ONLY_CONSUMER_CODE)
# ).
print("Your rollback is in the lobby waiting for an elevator, hold for 10 seconds...")
time.sleep(10)

# Snapshot the events count before the rollback run.
PRE_ROLLBACK_COUNT = int(_athena_scalar(
    f"SELECT count(*) FROM {DATABASE_NAME}.{EVENTS_TABLE}"
))
print(f"events_iceberg pre-rollback count: {PRE_ROLLBACK_COUNT}")

PRODUCER_BATCH_3 = f"batch-{int(time.time())}-3"
PRODUCER_RUN_BATCHES.append(PRODUCER_BATCH_3)
run_producer(PRODUCER_BATCH_3)
print(f"Rollback producer batch emitted: {PRODUCER_BATCH_3}")
print("Drain wait, ~60 seconds...")
time.sleep(60)


In [ ]:
# NBVAL_SKIP
# Checkpoint 5: events_iceberg grew by 100; no PARSE_ERROR in Lambda logs.

def validator_5(session) -> CheckpointResult:
    try:
        post = int(_athena_scalar(
            f"SELECT count(*) FROM {DATABASE_NAME}.{EVENTS_TABLE}"
        ))
    except Exception as exc:
        return CheckpointResult("error", error_reason=f"Athena query failed: {exc}")
    delta = post - PRE_ROLLBACK_COUNT
    if delta != 100:
        return CheckpointResult("fail", error_reason=(
            f"events_iceberg grew by {delta} rows, expected 100. "
            f"Wait another 30s and re-run if you suspect the drain is incomplete."
        ))

    # Scan the most recent Lambda log stream for PARSE_ERROR.
    logs_client = boto3.client(
        "logs",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    log_group = f"/aws/lambda/{CONSUMER_FN_NAME}"
    try:
        streams = logs_client.describe_log_streams(
            logGroupName=log_group, orderBy="LastEventTime", descending=True, limit=5,
        )["logStreams"]
    except logs_client.exceptions.ResourceNotFoundException:
        return CheckpointResult("error", error_reason=f"Log group {log_group} missing")
    parse_errors = 0
    for s in streams:
        try:
            ev = logs_client.filter_log_events(
                logGroupName=log_group,
                logStreamNames=[s["logStreamName"]],
                filterPattern="PARSE_ERROR",
                limit=50,
            )
            parse_errors += len(ev.get("events", []))
        except ClientError:
            continue
    if parse_errors:
        return CheckpointResult("fail", error_reason=(
            f"Found {parse_errors} PARSE_ERROR log lines in the rollback consumer. "
            f"The v1-only parser should tolerate v2 events silently (BACKWARD compatibility)."
        ))
    return CheckpointResult("pass")


check(5, validator_5)


<details><summary>Hint 1 (nudge)</summary>

BACKWARD compatibility means v1 readers can read v2 events without crashing; the extra field is simply ignored. Your rollback consumer does not need to know v2 exists.

</details>

<details><summary>Hint 2 (direction)</summary>

Replace the function code with the v1-only parser; the event-source mapping does not change. Re-run the producer (which still emits both shapes) and watch the v1-only consumer process them without errors. The new `referrer_domain` field is silently dropped because the v1 parser never references it.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
lambda_client.update_function_code(
    FunctionName=CONSUMER_FN_NAME,
    ZipFile=_make_zip(V1_ONLY_CONSUMER_CODE),
)
```

</details>


**Wallet check.** The rollback producer and drain adds another ~15 cents in Iceberg writes. Lambda still free. Kinesis shard-hour for the rollback portion: about another half cent. Session total so far: about 49 cents.

In [ ]:
# NBVAL_SKIP
# Cleanup. Critical-first (Kinesis stream tears down first), then ESM,
# Lambda, schema, registry, Iceberg tables, IAM role, Athena workgroup,
# Glue database, S3 bucket.

# An inline adapter handles the cleanup types labrun-checks does not yet
# cover natively (athena_workgroup, glue_schema, glue_registry, iceberg_table).
from labrun_checks.adapters.aws import AwsCleanupAdapter


class _LabAdapter(AwsCleanupAdapter):
    def delete_resource(self, credentials, resource):
        if resource.type == "athena_workgroup":
            client = boto3.client(
                "athena", region_name=resource.region,
                **{k: v for k, v in credentials.items() if v},
            )
            try:
                client.delete_work_group(
                    WorkGroup=resource.id, RecursiveDeleteOption=True,
                )
            except client.exceptions.InvalidRequestException:
                pass
            return
        if resource.type == "glue_schema":
            client = boto3.client(
                "glue", region_name=resource.region,
                **{k: v for k, v in credentials.items() if v},
            )
            try:
                client.delete_schema(
                    SchemaId={"RegistryName": REGISTRY_NAME, "SchemaName": resource.id}
                )
            except client.exceptions.EntityNotFoundException:
                pass
            return
        if resource.type == "glue_registry":
            client = boto3.client(
                "glue", region_name=resource.region,
                **{k: v for k, v in credentials.items() if v},
            )
            try:
                client.delete_registry(RegistryId={"RegistryName": resource.id})
            except client.exceptions.EntityNotFoundException:
                pass
            return
        if resource.type == "iceberg_table":
            db, table = resource.id.split(".", 1)
            client = boto3.client(
                "glue", region_name=resource.region,
                **{k: v for k, v in credentials.items() if v},
            )
            try:
                client.delete_table(DatabaseName=db, Name=table)
            except client.exceptions.EntityNotFoundException:
                pass
            # Also clear the table's data prefix from S3.
            s3c = boto3.client(
                "s3", region_name=resource.region,
                **{k: v for k, v in credentials.items() if v},
            )
            prefix = f"{table}/"
            try:
                while True:
                    listing = s3c.list_objects_v2(Bucket=BUCKET_NAME, Prefix=prefix)
                    objs = listing.get("Contents", [])
                    if not objs:
                        break
                    s3c.delete_objects(
                        Bucket=BUCKET_NAME,
                        Delete={"Objects": [{"Key": o["Key"]} for o in objs]},
                    )
                    if not listing.get("IsTruncated"):
                        break
            except ClientError:
                pass
            return
        return super().delete_resource(credentials, resource)

    def describe_resource(self, credentials, resource):
        if resource.type in ("athena_workgroup", "glue_schema", "glue_registry", "iceberg_table"):
            # Best-effort: treat as gone after delete_resource succeeds.
            return False
        return super().describe_resource(credentials, resource)


# Run cleanup.
result = run_cleanup(CLEANUP_MANIFEST, adapter=_LabAdapter())

print()
print("Cleanup complete.")
critical_count = sum(1 for r in CLEANUP_MANIFEST if r.critical)
standard_count = len(CLEANUP_MANIFEST) - critical_count
print(f"Critical resources destroyed: {critical_count}")
print(f"Standard resources destroyed: {standard_count}")
failed = len(result.warnings) + len(result.orphans)
print(f"Resources that failed to delete: {failed}")
if failed > 0:
    print("If failed > 0, your AWS account may still incur charges. Resolve before closing this notebook.")
    for w in result.warnings:
        print(f"  WARNING: {w}")
    for o in result.orphans:
        print(f"  ORPHAN: {o}")

cleanup(status=result.status)

if result.status == "dirty":
    sys.exit(1)


**Session total: $0.60 to $1.20.** Glue Iceberg writes were the largest line item. Kinesis shard-hour added about 3 cents across the 90-minute session. The Schema Registry was free at this volume. Lambda was free. Athena scans for the validators added pennies. If you ran a debug round of the producer or hit the InvalidParameterValueException retry loop on the Lambda deploy, the upper bound applies. Check your AWS Billing console in 24 hours to confirm.

## These are not graded. They are for you.

**1. Compatibility modes.** Walk through what BACKWARD, FORWARD, and FULL each guarantee. If you were the consumer team and the producer team shipped fields more often than you could keep up, which mode would you ask the registry to enforce, and why?

**2. The rollback test you did not write.** The lab tested rollback to a v1-only parser. In production you may also need to roll forward: the consumer is on v2 code, but you discover a bug specific to v2 events, so you want to skip v2 records until the fix lands. How would you implement that without touching the schema registry?

## Exam-Style Practice

**Question 1.** Your producer team wants to drop a field that an existing consumer relies on. You own the schema registry and you need to pick the compatibility mode that allows this safely. Which mode do you set?

A) BACKWARD, because it allows new schemas to drop fields the consumer reads.

B) FORWARD, because it allows new producers to evolve in ways future consumers can already handle.

C) FULL, because it combines BACKWARD plus FORWARD and is therefore the safest pick.

D) NONE, because dropping a field is not a schema evolution operation and should bypass the registry.

<details><summary>Show answer</summary>

**B**.

- A) Wrong. BACKWARD means new schemas must let old consumers keep reading. Dropping a field old consumers rely on breaks them; BACKWARD rejects this change.
- B) Right. FORWARD means new producers can evolve in ways the existing (future) consumers can handle. Dropping a field is allowed if every consumer already tolerates its absence (typically because the consumer treats it as optional). FORWARD is the right mode when producers move first and consumers must accept the change.
- C) Wrong. FULL combines BACKWARD and FORWARD; it is the strictest mode. A drop that fails BACKWARD would also fail FULL.
- D) Wrong. The registry exists precisely to govern these changes. Bypassing it removes the safety the registry provides; if you do not want the registry's guarantees you should not be using it.

</details>

**Question 2.** Your pipeline reads Kinesis events that carry an explicit `schema_version` field on every record. A new producer team is onboarding and they pitch encoding events with Avro plus the Glue Schema Registry SerDe library (the binary magic-byte + schema-id header on each record). Which is the single best reason to accept their pitch?

A) Avro is faster than JSON to parse, so per-record CPU cost goes down.

B) Avro is more compact than JSON, so PUT payload unit cost goes down.

C) Avro with the SerDe library carries the schema ID in a binary header, so consumers can reject unknown schemas before the JSON parser even runs.

D) Avro is the only format the Glue Schema Registry supports.

<details><summary>Show answer</summary>

**C**.

- A) Partially true but minor. Avro parsing is faster than JSON parsing per record, but at lab volumes (or even mid-size production) the per-record CPU difference is below the noise floor of Lambda's cold start and the network round trip. It is not the strongest reason.
- B) Partially true at high volume. Avro is more compact, so PUT payload units shrink for large records. But for the typical 200-byte clickstream event, both formats land inside a single PUT payload unit (25 KB), so the cost difference is zero at this scale.
- C) Right. The binary header carries the schema ID directly. The consumer reads 17 bytes (magic byte + 16-byte UUID schema id), looks up the schema in the registry, and either deserializes with that schema or rejects the record before any payload parsing. JSON with an embedded `schema_version` field requires the consumer to JSON-parse the entire payload before it can route the record. The header-based approach is the architectural win: schema validation happens before deserialization.
- D) Wrong. The Glue Schema Registry supports Avro, JSON Schema, and Protobuf. Avro is one of three; it is not the only option.

</details>
